# 📊 Comparaison des modèles Teacher vs Student

Ce notebook compare les performances du modèle DistilBERT fine-tuné (teacher) et du modèle distillé (student) entraîné via knowledge distillation.

## 1️⃣ Chargement des dépendances et configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Paramètres d'affichage
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)


## 2️⃣ Chargement du dataset de test

In [ ]:
TEST_PATH = "./data/test.csv"

assert os.path.exists(TEST_PATH), f"Fichier test introuvable : {TEST_PATH}"
df_test = pd.read_csv(TEST_PATH)
df_test.head()

## 3️⃣ Chargement des modèles Teacher et Student

In [ ]:
teacher_model = tf.keras.models.load_model("notebooks/models/distilber_sentiment")
student_model = tf.keras.models.load_model("notebooks/models/distilber_sentiment/student_model")

## 4️⃣ Préparation des données pour l'inférence

In [ ]:
from transformers import DistilBertTokenizerFast

# Tokenizer (doit correspondre à celui utilisé lors de l'entraînement)
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def encode_texts(texts, max_len=256):
    tokens = tokenizer(
        texts.tolist(),
        max_length=max_len,
        padding="max_length",
        truncation=True,
        return_tensors="tf"
    )
    return tokens

x_test_tokens = encode_texts(df_test["clean_text"])
y_test = df_test["label"].values


## 5️⃣ Prédictions des deux modèles

In [ ]:
teacher_preds = teacher_model.predict(x_test_tokens).flatten()
student_preds = student_model.predict(x_test_tokens).flatten()

teacher_labels = (teacher_preds >= 0.5).astype(int)
student_labels = (student_preds >= 0.5).astype(int)


## 6️⃣ Rapport de classification

In [ ]:
print("📘 Teacher model:")
print(classification_report(y_test, teacher_labels))

print("📗 Student model:")
print(classification_report(y_test, student_labels))


## 7️⃣ Matrices de confusion

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, labels, title in zip(
    axes,
    [teacher_labels, student_labels],
    ["Teacher Model", "Student Model"]
):
    cm = confusion_matrix(y_test, labels)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Prédit")
    ax.set_ylabel("Réel")

plt.tight_layout()
plt.show()


## 8️⃣ Conclusion

Le modèle distillé (student) offre une **bonne performance générale** tout en étant **plus léger et rapide** que le modèle original (teacher).

- Le modèle Teacher reste légèrement supérieur en précision.
- Le modèle Student est plus efficace pour le déploiement en production ou embarqué.

📌 Ce compromis entre taille et performance est l'objectif principal de la distillation de connaissances.
